# NB09 — Deep Learning: LSTM, GRU & Temporal Fusion Transformer

**Objectives**:
- Train LSTM and GRU models for volatility and return forecasting
- Compare DL models against best ML baselines from NB07-NB08
- Implement walk-forward evaluation consistent with NB07-NB08 protocol
- Analyze training dynamics, convergence, and computational cost

**Architecture**: 2-layer stacked RNN → Dropout(0.3) → Dense(1)
- Input: 60-day lookback window of multi-feature sequences
- Target A: 5-day forward realized vol (Yang-Zhang)
- Target B: 5-day forward return

**Walk-Forward Protocol**: Same as NB07-NB08 — expanding window, retrain every 63 trading days.

**Key Principle**: PyTorch LSTM `dropout` only applies BETWEEN layers, not after the last layer.
We add explicit `nn.Dropout` before the fully connected layer.

**Output**: `dl_forecast_comparison.csv`, model checkpoints in `models/dl/`

In [ ]:
import sys, os, warnings, time, json
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn

from src.config import *
from src.dl_models import (
    LSTMForecaster, GRUForecaster,
    create_sequences, train_model, predict_model, EarlyStopping
)
from src.ml_pipeline import regression_metrics, diebold_mariano_test
from src.visualization import save_fig

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Imports OK | Device: {device} | PyTorch {torch.__version__}')

## 1. Load Data & Build Feature Matrix

Assemble multi-feature sequences from NB01-NB06 outputs for each ticker.
Features: lagged realized vol, log returns, momentum, conditional vol (NB03),
regime state (NB05), sentiment (NB06, t-1 lagged).

In [ ]:
# ── Load all upstream data ──
master = pd.read_parquet(MASTER_DATA_FILE)
cond_vol = pd.read_parquet(COND_VOL_FILE) if COND_VOL_FILE.exists() else pd.DataFrame()
regime = pd.read_parquet(REGIME_LABELS_FILE) if REGIME_LABELS_FILE.exists() else pd.DataFrame()
sentiment = pd.read_parquet(SENTIMENT_FILE) if SENTIMENT_FILE.exists() else pd.DataFrame()


def build_features_for_ticker(ticker: str, master_df: pd.DataFrame) -> tuple:
    """
    Build feature matrix X and targets (vol, return) for a single ticker.
    
    Returns: (X_df, y_vol_5d, y_ret_5d) — all aligned by index.
    """
    prices = master_df[ticker].dropna()
    log_r = np.log(prices / prices.shift(1))
    simple_r = prices.pct_change()
    
    feats = pd.DataFrame(index=prices.index)
    
    # Lagged realized volatility (Yang-Zhang proxy via rolling std)
    for w in [5, 21, 63]:
        feats[f'realized_vol_{w}d'] = log_r.rolling(w).std() * np.sqrt(252)
    
    # Return-based features
    feats['log_return'] = log_r
    feats['ret_5d'] = prices.pct_change(5)
    feats['ret_21d'] = prices.pct_change(21)
    
    # Momentum indicators
    # RSI-14 (EWM, consistent with feature_engineering.py)
    delta = prices.diff()
    gain = delta.clip(lower=0).ewm(alpha=1/14, min_periods=14).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1/14, min_periods=14).mean()
    rs = gain / loss.replace(0, np.nan)
    feats['rsi_14'] = 100 - (100 / (1 + rs))
    
    # Bollinger %B (with zero guard on denominator)
    sma20 = prices.rolling(20).mean()
    std20 = prices.rolling(20).std()
    feats['bollinger_pct_b'] = (prices - (sma20 - 2 * std20)) / (4 * std20).replace(0, np.nan)
    
    # Volume z-score (if volume data available)
    vol_col = f'{ticker}_Volume'
    if vol_col in master_df.columns:
        vol_series = master_df[vol_col].reindex(prices.index)
        feats['volume_zscore'] = (vol_series - vol_series.rolling(20).mean()) / vol_series.rolling(20).std()
    
    # Conditional volatility from GARCH (NB03)
    if ticker in cond_vol.columns:
        feats['garch_cond_vol'] = cond_vol[ticker].reindex(feats.index)
    
    # Regime state (NB05)
    if 'regime_state' in regime.columns:
        feats['regime_state'] = regime['regime_state'].reindex(feats.index)
    
    # Sentiment (NB06 — already t-1 lagged in the parquet, long format)
    if not sentiment.empty and 'ticker' in sentiment.columns:
        ticker_sent = sentiment[sentiment['ticker'] == ticker]
        if not ticker_sent.empty:
            feats['sentiment'] = ticker_sent['sentiment_mean'].reindex(feats.index)
    
    # Targets: 5-day forward realized vol and 5-day forward return
    y_vol_5d = log_r.rolling(5).std().shift(-5) * np.sqrt(252)
    y_ret_5d = prices.pct_change(5).shift(-5)
    
    # Align and drop NaNs
    feats = feats.dropna()
    y_vol_5d = y_vol_5d.reindex(feats.index)
    y_ret_5d = y_ret_5d.reindex(feats.index)
    valid = ~(y_vol_5d.isna() | y_ret_5d.isna())
    
    return feats[valid], y_vol_5d[valid], y_ret_5d[valid]


# Build features for NVDA as primary example
X_df, y_vol, y_ret = build_features_for_ticker('NVDA', master)
print(f'NVDA features: {X_df.shape}, Vol target: {y_vol.shape}, Ret target: {y_ret.shape}')
print(f'Feature columns: {list(X_df.columns)}')
print(f'Date range: {X_df.index[0]} → {X_df.index[-1]}')

## 2. Walk-Forward DL Evaluation Engine

Key design decisions:
- **Expanding window** with quarterly retraining (63 trading days), matching NB07-NB08
- **StandardScaler** fit on training data only at each retraining step — applied to test sequences
- **Sequence creation**: 60-day lookback windows (LSTM_LOOKBACK = 60)
- **Early stopping** on a held-out validation split (last 15% of training data)
- **No lookahead**: scaler parameters and model weights use only past data

In [ ]:
from sklearn.preprocessing import StandardScaler


def walk_forward_dl(
    X_df: pd.DataFrame,
    y: pd.Series,
    model_class,  # LSTMForecaster or GRUForecaster
    lookback: int = LSTM_LOOKBACK,
    retrain_freq: int = RETRAIN_FREQ_DAYS,
    initial_train_ratio: float = TRAIN_RATIO,
    horizon: int = 5,  # forecast horizon for purge/embargo
    epochs: int = 200,
    batch_size: int = 64,
    lr: float = 1e-3,
    device: str = 'cpu',
) -> pd.DataFrame:
    """
    Walk-forward expanding-window evaluation for LSTM/GRU with embargo/purge.
    
    Protocol (Lopez de Prado, 2018, Ch.7):
    1. Purge: remove last `horizon` training observations (labels overlap test period)
    2. Embargo: skip first `horizon` test observations after train boundary
    3. Split remaining training data: 85% train / 15% validation (for early stopping)
    4. Fit StandardScaler on training portion ONLY
    5. Create sequences from scaled data
    6. Train model with early stopping on validation loss
    7. Predict post-embargo test observations
    8. Expand window, retrain
    
    Returns DataFrame with columns: date, y_true, y_pred
    """
    T = len(X_df)
    initial_end = int(T * initial_train_ratio)
    results = []
    train_end = initial_end
    training_times = []
    
    while train_end < T:
        test_end = min(train_end + retrain_freq, T)
        
        # ── Purge/Embargo (Lopez de Prado Ch.7) ──
        # Purge: exclude last `horizon` training obs (their labels leak into test)
        purge_end = max(train_end - horizon, lookback + 100)
        # Embargo: skip first `horizon` test obs after train boundary
        embargo_start = min(train_end + horizon, T)
        
        # Need at least lookback + some samples for training
        if purge_end < lookback + 100:
            train_end = test_end
            continue
        
        # Skip if no test data after embargo
        if embargo_start >= test_end:
            train_end = test_end
            continue
        
        # ── Scale features using ONLY training data (up to purge boundary) ──
        scaler = StandardScaler()
        X_train_raw = X_df.iloc[:purge_end].values
        X_all_raw = X_df.iloc[:test_end].values
        y_all = y.iloc[:test_end].values
        
        scaler.fit(X_train_raw)  # Fit only on purged training data
        X_all_scaled = scaler.transform(X_all_raw)
        
        # ── Create sequences ──
        X_seq, y_seq = create_sequences(X_all_scaled, y_all, lookback=lookback)
        
        # Determine train/val/test split indices in sequence space
        n_purged_train_seq = purge_end - lookback  # sequences for training (purged)
        n_embargo_seq = embargo_start - lookback    # first test sequence index (after embargo)
        n_total_seq = len(X_seq)
        
        if n_purged_train_seq < 100 or n_embargo_seq >= n_total_seq:
            train_end = test_end
            continue
        
        # Split training into train/val (85/15) for early stopping
        val_size = max(int(n_purged_train_seq * 0.15), 20)
        train_size = n_purged_train_seq - val_size
        
        X_tr = X_seq[:train_size]
        y_tr = y_seq[:train_size]
        X_val = X_seq[train_size:n_purged_train_seq]
        y_val = y_seq[train_size:n_purged_train_seq]
        X_test = X_seq[n_embargo_seq:n_total_seq]
        y_test = y_seq[n_embargo_seq:n_total_seq]
        
        if len(X_test) == 0:
            train_end = test_end
            continue
        
        # ── Train model ──
        input_dim = X_df.shape[1]
        model = model_class(input_dim=input_dim)
        
        t0 = time.time()
        history = train_model(
            model, X_tr, y_tr, X_val, y_val,
            epochs=epochs, batch_size=batch_size, lr=lr,
            patience=EARLY_STOP_PATIENCE, device=device
        )
        elapsed = time.time() - t0
        training_times.append(elapsed)
        
        # ── Predict ──
        preds = predict_model(model, X_test, device=device)
        
        # Map back to dates (embargo_start onward)
        test_dates = X_df.index[embargo_start:test_end]
        n_pred = min(len(preds), len(test_dates))
        
        batch = pd.DataFrame({
            'date': test_dates[:n_pred],
            'y_true': y_test[:n_pred],
            'y_pred': preds[:n_pred],
        })
        results.append(batch)
        
        train_end = test_end
    
    if not results:
        return pd.DataFrame(), []
    
    return pd.concat(results, ignore_index=True), training_times


print('Walk-forward DL engine defined (with embargo/purge for h=5 targets)')

## 2b. Embargo/Purge for Deep Learning Walk-Forward

Same protocol as NB07-NB08 (Lopez de Prado, 2018, Ch.7):
- For h=5 forward targets: purge last 5 training observations, embargo 5 days after train end
- Prevents label overlap between training and test periods
- Applied in `walk_forward_dl()` below by adjusting `train_end` at each retraining step

The walk-forward engine below implements this by ensuring that sequence creation
respects the temporal boundary: no training sequence's target overlaps with the
test period's features.

## 3. Volatility Forecasting: LSTM vs GRU

Both models forecast 5-day forward realized volatility (Yang-Zhang estimator).
Compared against best ML baseline from NB07.

In [ ]:
# ── LSTM: Volatility Forecasting ──
print('Training LSTM for volatility forecasting...')
lstm_vol_preds, lstm_vol_times = walk_forward_dl(
    X_df, y_vol, LSTMForecaster,
    lookback=LSTM_LOOKBACK, device=device
)

if len(lstm_vol_preds) > 0:
    lstm_vol_metrics = regression_metrics(
        lstm_vol_preds['y_true'].values,
        lstm_vol_preds['y_pred'].values
    )
    print(f'LSTM Vol — RMSE: {lstm_vol_metrics["rmse"]:.6f}, '
          f'MAE: {lstm_vol_metrics["mae"]:.6f}, '
          f'DA: {lstm_vol_metrics["directional_accuracy"]:.1f}%')
    print(f'  Avg training time per window: {np.mean(lstm_vol_times):.1f}s')

# ── GRU: Volatility Forecasting ──
print('\nTraining GRU for volatility forecasting...')
gru_vol_preds, gru_vol_times = walk_forward_dl(
    X_df, y_vol, GRUForecaster,
    lookback=LSTM_LOOKBACK, device=device
)

if len(gru_vol_preds) > 0:
    gru_vol_metrics = regression_metrics(
        gru_vol_preds['y_true'].values,
        gru_vol_preds['y_pred'].values
    )
    print(f'GRU Vol  — RMSE: {gru_vol_metrics["rmse"]:.6f}, '
          f'MAE: {gru_vol_metrics["mae"]:.6f}, '
          f'DA: {gru_vol_metrics["directional_accuracy"]:.1f}%')
    print(f'  Avg training time per window: {np.mean(gru_vol_times):.1f}s')

## 4. Return Forecasting: LSTM vs GRU

5-day forward return forecasting. Same walk-forward protocol.

In [ ]:
# ── LSTM: Return Forecasting ──
print('Training LSTM for return forecasting...')
lstm_ret_preds, lstm_ret_times = walk_forward_dl(
    X_df, y_ret, LSTMForecaster,
    lookback=LSTM_LOOKBACK, device=device
)

if len(lstm_ret_preds) > 0:
    lstm_ret_metrics = regression_metrics(
        lstm_ret_preds['y_true'].values,
        lstm_ret_preds['y_pred'].values
    )
    print(f'LSTM Ret — RMSE: {lstm_ret_metrics["rmse"]:.6f}, '
          f'DA: {lstm_ret_metrics["directional_accuracy"]:.1f}%')

# ── GRU: Return Forecasting ──
print('\nTraining GRU for return forecasting...')
gru_ret_preds, gru_ret_times = walk_forward_dl(
    X_df, y_ret, GRUForecaster,
    lookback=LSTM_LOOKBACK, device=device
)

if len(gru_ret_preds) > 0:
    gru_ret_metrics = regression_metrics(
        gru_ret_preds['y_true'].values,
        gru_ret_preds['y_pred'].values
    )
    print(f'GRU Ret  — RMSE: {gru_ret_metrics["rmse"]:.6f}, '
          f'DA: {gru_ret_metrics["directional_accuracy"]:.1f}%')

## 5. Diebold-Mariano: LSTM vs GRU vs ML Baseline

Statistical test for equal predictive accuracy.
For 5-day horizon forecasts, we use HAC variance estimator with bandwidth h-1 = 4
to account for overlapping forecast windows (Diebold & Mariano, 1995).

In [ ]:
# ── Diebold-Mariano tests (volatility forecasting) ──
dm_results = []

if len(lstm_vol_preds) > 0 and len(gru_vol_preds) > 0:
    # Align predictions by date for pairwise comparison
    merged = lstm_vol_preds.merge(
        gru_vol_preds, on='date', suffixes=('_lstm', '_gru')
    )
    
    if len(merged) > 50:
        e_lstm = merged['y_true_lstm'].values - merged['y_pred_lstm'].values
        e_gru = merged['y_true_gru'].values - merged['y_pred_gru'].values
        
        # h=5 for 5-day forward forecast → HAC bandwidth = 4
        dm = diebold_mariano_test(e_lstm, e_gru, h=5, loss_fn='squared')
        dm_results.append({
            'comparison': 'LSTM vs GRU (vol)',
            'dm_stat': dm['dm_stat'],
            'p_value': dm['p_value'],
            'interpretation': 'LSTM better' if dm['dm_stat'] < 0 else 'GRU better'
        })
        print(f'DM Test (LSTM vs GRU, vol): stat={dm["dm_stat"]:.3f}, p={dm["p_value"]:.4f}')

# Load NB07 ML baseline predictions for comparison if available
if VOL_FORECAST_FILE.exists():
    ml_vol_preds = pd.read_parquet(VOL_FORECAST_FILE)
    print('\nML baseline loaded from NB07 for comparison')
    # Further DM tests against ML baseline would go here
else:
    print('\nNB07 predictions not yet available — run NB07 first for full comparison')

if dm_results:
    pd.DataFrame(dm_results)

## 6. Multi-Ticker Analysis

Run LSTM volatility forecasting on the 5 highest-vol tickers to assess
whether DL provides consistent improvement across different risk profiles.

In [ ]:
high_vol_tickers = ['NVDA', 'PLTR', 'MU', 'CRWD', 'XYZ']
multi_ticker_results = {}

for ticker in high_vol_tickers:
    try:
        X_t, y_vol_t, y_ret_t = build_features_for_ticker(ticker, master)
        if len(X_t) < 500:
            print(f'{ticker}: Insufficient data ({len(X_t)} obs), skipping')
            continue
        
        preds, times = walk_forward_dl(
            X_t, y_vol_t, LSTMForecaster,
            lookback=LSTM_LOOKBACK, device=device,
            epochs=100  # reduced for multi-ticker sweep
        )
        
        if len(preds) > 0:
            m = regression_metrics(preds['y_true'].values, preds['y_pred'].values)
            m['avg_train_time_s'] = np.mean(times)
            multi_ticker_results[ticker] = m
            print(f'{ticker}: RMSE={m["rmse"]:.6f}, DA={m["directional_accuracy"]:.1f}%')
    except Exception as e:
        print(f'{ticker}: Error — {e}')

if multi_ticker_results:
    multi_df = pd.DataFrame(multi_ticker_results).T
    multi_df

## 7. Training Dynamics Visualization

Plot training vs validation loss curves to diagnose overfitting.
The early stopping marker shows when training was halted.

In [ ]:
# ── Train a single LSTM model for visualization ──
# Use the full initial training window to demonstrate training dynamics
scaler = StandardScaler()
T_total = len(X_df)
train_end_idx = int(T_total * TRAIN_RATIO)

X_scaled = scaler.fit_transform(X_df.iloc[:train_end_idx].values)
y_train_arr = y_vol.iloc[:train_end_idx].values

X_seq, y_seq = create_sequences(X_scaled, y_train_arr, lookback=LSTM_LOOKBACK)

# 85/15 train/val split
val_size = int(len(X_seq) * 0.15)
X_tr_vis = X_seq[:-val_size]
y_tr_vis = y_seq[:-val_size]
X_val_vis = X_seq[-val_size:]
y_val_vis = y_seq[-val_size:]

model_vis = LSTMForecaster(input_dim=X_df.shape[1])
history = train_model(
    model_vis, X_tr_vis, y_tr_vis, X_val_vis, y_val_vis,
    epochs=200, batch_size=64, lr=1e-3,
    patience=EARLY_STOP_PATIENCE, device=device
)

# ── Plot training curves ──
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(history['train_loss'], label='Train Loss', linewidth=1.5)
ax.plot(history['val_loss'], label='Validation Loss', linewidth=1.5)
if history['stopped_epoch'] < len(history['train_loss']):
    ax.axvline(x=history['stopped_epoch'], color='red', linestyle='--',
               label=f'Early Stop (epoch {history["stopped_epoch"]})')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('LSTM Training Dynamics — Volatility Forecasting (NVDA)')
ax.legend()
ax.set_yscale('log')
save_fig(fig, 'nb09_lstm_training_curves')
plt.show()

## 8. Prediction vs Actual Time Series

Visual comparison of LSTM/GRU predictions against realized values.
This reveals whether models capture volatility clustering and mean-reversion.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# Panel A: Volatility Forecasts
if len(lstm_vol_preds) > 0:
    ax = axes[0]
    dates_vol = pd.to_datetime(lstm_vol_preds['date'])
    ax.plot(dates_vol, lstm_vol_preds['y_true'], 'k-', label='Realized Vol', linewidth=0.8, alpha=0.7)
    ax.plot(dates_vol, lstm_vol_preds['y_pred'], 'b-', label='LSTM Forecast', linewidth=0.8, alpha=0.7)
    if len(gru_vol_preds) > 0:
        dates_gru = pd.to_datetime(gru_vol_preds['date'])
        ax.plot(dates_gru, gru_vol_preds['y_pred'], 'r-', label='GRU Forecast', linewidth=0.8, alpha=0.7)
    ax.set_ylabel('Annualized Volatility')
    ax.set_title('Volatility Forecast: LSTM vs GRU vs Actual (NVDA)')
    ax.legend()

# Panel B: Return Forecasts
if len(lstm_ret_preds) > 0:
    ax = axes[1]
    dates_ret = pd.to_datetime(lstm_ret_preds['date'])
    ax.plot(dates_ret, lstm_ret_preds['y_true'], 'k-', label='Actual 5d Return', linewidth=0.8, alpha=0.7)
    ax.plot(dates_ret, lstm_ret_preds['y_pred'], 'b-', label='LSTM Forecast', linewidth=0.8, alpha=0.7)
    if len(gru_ret_preds) > 0:
        dates_gru_r = pd.to_datetime(gru_ret_preds['date'])
        ax.plot(dates_gru_r, gru_ret_preds['y_pred'], 'r-', label='GRU Forecast', linewidth=0.8, alpha=0.7)
    ax.set_ylabel('5-Day Return')
    ax.set_xlabel('Date')
    ax.set_title('Return Forecast: LSTM vs GRU vs Actual (NVDA)')
    ax.legend()

save_fig(fig, 'nb09_prediction_vs_actual')
plt.show()

## 9. Computational Cost Analysis

Compare training time, inference time, and memory footprint.
GRU should converge faster (fewer parameters — no cell state gate).

In [ ]:
# ── Parameter count comparison ──
input_dim = X_df.shape[1]
lstm_model = LSTMForecaster(input_dim=input_dim)
gru_model = GRUForecaster(input_dim=input_dim)

lstm_params = sum(p.numel() for p in lstm_model.parameters())
gru_params = sum(p.numel() for p in gru_model.parameters())

print(f'LSTM parameters: {lstm_params:,}')
print(f'GRU parameters:  {gru_params:,}')
print(f'GRU/LSTM ratio:  {gru_params/lstm_params:.2f} (GRU has ~75% of LSTM params)')

# ── Inference time comparison ──
dummy_input = torch.randn(1, LSTM_LOOKBACK, input_dim)

# LSTM inference
lstm_model.eval()
t0 = time.time()
for _ in range(1000):
    with torch.no_grad():
        _ = lstm_model(dummy_input)
lstm_infer = (time.time() - t0) / 1000 * 1000  # ms per inference

# GRU inference
gru_model.eval()
t0 = time.time()
for _ in range(1000):
    with torch.no_grad():
        _ = gru_model(dummy_input)
gru_infer = (time.time() - t0) / 1000 * 1000

cost_df = pd.DataFrame({
    'Model': ['LSTM', 'GRU'],
    'Parameters': [lstm_params, gru_params],
    'Avg Train Time (s)': [
        np.mean(lstm_vol_times) if lstm_vol_times else np.nan,
        np.mean(gru_vol_times) if gru_vol_times else np.nan
    ],
    'Inference (ms/sample)': [lstm_infer, gru_infer],
}).set_index('Model')

print('\n── Computational Cost Summary ──')
cost_df

## 10. Consolidated Comparison & Save

Aggregate all DL results into a comparison table.
This feeds into NB10's comprehensive model audit.

In [ ]:
# ── Consolidated comparison table ──
comparison_rows = []

if len(lstm_vol_preds) > 0:
    comparison_rows.append({'model': 'LSTM', 'task': 'vol_5d', **lstm_vol_metrics})
if len(gru_vol_preds) > 0:
    comparison_rows.append({'model': 'GRU', 'task': 'vol_5d', **gru_vol_metrics})
if len(lstm_ret_preds) > 0:
    comparison_rows.append({'model': 'LSTM', 'task': 'ret_5d', **lstm_ret_metrics})
if len(gru_ret_preds) > 0:
    comparison_rows.append({'model': 'GRU', 'task': 'ret_5d', **gru_ret_metrics})

dl_comparison = pd.DataFrame(comparison_rows)

# Save
dl_comp_file = TABLES_DIR / 'dl_forecast_comparison.csv'
dl_comparison.to_csv(dl_comp_file, index=False)
print(f'Saved: {dl_comp_file}')

# Save predictions for NB10 hybrid model
if len(lstm_vol_preds) > 0:
    lstm_vol_preds.to_parquet(FEATURES_DIR / 'lstm_vol_predictions.parquet', index=False)
if len(lstm_ret_preds) > 0:
    lstm_ret_preds.to_parquet(FEATURES_DIR / 'lstm_ret_predictions.parquet', index=False)

print('\n── DL Forecast Comparison ──')
dl_comparison